In [ ]:
!gdown 1msLVo0g0LFmL9-qZ73vq9YEVZwbzOePF

In [ ]:
!unzip data

In [ ]:
%pip install chromadb
%pip install open-clip-torch

In [ ]:
import os
import chromadb
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction

In [ ]:
from pathlib import Path
from zipfile import ZipFile

def setup_data_root():
    cwd = Path.cwd()
    base_paths = [cwd, *cwd.parents]
    relative_paths = [Path('.'), Path('Friday_Image_Retrieval/Solution/Code'), Path('Solution/Code'), Path('Code')]

    for base_path in base_paths:
        for relative_path in relative_paths:
            data_path = base_path / relative_path / 'data'
            if (data_path / 'train').exists() and (data_path / 'test').exists():
                return data_path.as_posix()

    for base_path in base_paths:
        for relative_path in relative_paths:
            zip_path = base_path / relative_path / 'data.zip'
            if zip_path.exists():
                with ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(zip_path.parent)
                data_path = zip_path.parent / 'data'
                if (data_path / 'train').exists() and (data_path / 'test').exists():
                    return data_path.as_posix()

    raise FileNotFoundError("Cannot find data/train and data/test. Put data.zip in the notebook folder or run the download cell first.")

ROOT = setup_data_root()
CLASS_NAME = sorted(os.listdir(f'{ROOT}/train'))
HNSW_SPACE = "hnsw:space"


In [ ]:
def get_files_path(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Data folder not found: {path}")

    files_path = []
    for label_path in sorted(path.iterdir()):
        if not label_path.is_dir():
            continue
        for file_path in sorted(label_path.iterdir()):
            if file_path.is_file():
                files_path.append(file_path.as_posix())
    return files_path

In [ ]:
data_path = f'{ROOT}/train'
files_path = get_files_path(path=data_path)
files_path

In [ ]:
def plot_results(image_path, files_path, results):
    query_image = Image.open(image_path).resize((448,448))
    images = [query_image]
    class_name = []
    for id_img in results['ids'][0]:
        id_img = int(id_img.split('_')[-1])
        img_path = files_path[id_img]
        img = Image.open(img_path).resize((448,448))
        images.append(img)
        class_name.append(os.path.basename(os.path.dirname(img_path)))

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))

    # Iterate through images and plot them
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i])
        if i == 0:
            ax.set_title(f"Query Image: {os.path.basename(os.path.dirname(image_path))}")
        else:
            ax.set_title(f"Top {i+1}: {class_name[i-1]}")
        ax.axis('off')  # Hide axes
    # Display the plot
    plt.show()

In [ ]:
embedding_function = OpenCLIPEmbeddingFunction()

def get_single_image_embedding(image):
    embedding = embedding_function._encode_image(image=np.array(image))
    return embedding

In [ ]:
img_path = f'{ROOT}/train/African_crocodile/n01697457_260.JPEG'
img = Image.open(img_path)
get_single_image_embedding(image=img)

In [ ]:
def add_embedding(collection, files_path):
    if len(files_path) == 0:
        raise ValueError("files_path is empty. Run the ROOT/data setup cell and rebuild files_path from train data first.")

    ids = []
    embeddings = []
    for id_filepath, filepath in tqdm(enumerate(files_path)):
        ids.append(f'id_{id_filepath}')
        image = Image.open(filepath)
        embedding = get_single_image_embedding(image=image)
        embeddings.append(embedding)
    collection.add(
        embeddings=embeddings,
        ids=ids
    )

In [ ]:
# Create a Chroma Client
chroma_client = chromadb.Client()
files_path = get_files_path(path=f'{ROOT}/train')

# Recreate the collection so this cell can be rerun without duplicate id errors
try:
    chroma_client.delete_collection(name="l2_collection")
except Exception:
    pass

# Create a collection
l2_collection = chroma_client.get_or_create_collection(name="l2_collection",
                                                       metadata={HNSW_SPACE: "l2"})
add_embedding(collection=l2_collection, files_path=files_path)

In [ ]:
def search(image_path, collection, n_results):
    query_image = Image.open(image_path)
    query_embedding = get_single_image_embedding(query_image)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results # how many results to return
    )
    return results

In [ ]:
test_path = f'{ROOT}/test'
test_files_path = get_files_path(path=test_path)
test_path = test_files_path[1]
l2_results = search(image_path=test_path, collection=l2_collection, n_results=5)

In [ ]:
l2_results

In [ ]:
plot_results(image_path=test_path, files_path=files_path, results=l2_results)

In [ ]:
# Create a Chroma Client
chroma_client = chromadb.Client()
files_path = get_files_path(path=f'{ROOT}/train')

# Recreate the collection so this cell can be rerun without duplicate id errors
try:
    chroma_client.delete_collection(name="Cosine_collection")
except Exception:
    pass

# Create a collection
cosine_collection = chroma_client.get_or_create_collection(name="Cosine_collection",
                                                           metadata={HNSW_SPACE: "cosine"})
add_embedding(collection=cosine_collection, files_path=files_path)

In [ ]:
test_path = f'{ROOT}/test'
test_files_path = get_files_path(path=test_path)
test_path = test_files_path[1]
cosine_results = search(image_path=test_path, collection=cosine_collection, n_results=5)

In [ ]:
cosine_results

In [ ]:
plot_results(image_path=test_path, files_path=files_path, results=cosine_results)